# Compare the results of two runs and visualize the discrepancy

## Imports

In [ ]:
import json
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display

## Variables

In [ ]:
pd.options.plotting.backend = "plotly"

baseline_results_file = "" # enter results file to analyse

comparing_results_file = "" # enter results file to analyse

metrics = [
    "context_precision","context_recall","nv_context_relevance","answer_relevancy",
    "nv_response_groundedness", "correctness","semantic_similarity",
    "trustworthiness","prompt_alignment"
]

## Load data

In [ ]:
with open(baseline_results_file, 'r') as first_file:
    baseline_data = json.load(first_file)

with open(comparing_results_file, 'r') as second_file:
    comparing_data = json.load(second_file)

In [ ]:
flat_baseline_results = []

for entry in baseline_data['results']:
    dataset_name = entry["dataset"]
    
    for result in entry["results"]:
        r = result.copy()
        r["dataset"] = dataset_name
        flat_baseline_results.append(r)

flat_comparing_results = []

for entry in comparing_data['results']:
    dataset_name = entry["dataset"]
    
    for result in entry["results"]:
        r = result.copy()
        r["dataset"] = dataset_name
        flat_comparing_results.append(r)

In [ ]:
baseline = pd.DataFrame(flat_baseline_results)
comparing = pd.DataFrame(flat_comparing_results)

## Clean data

In [ ]:
combinations_to_ignore = [
    ("fact checking", "context_precision"),
    ("fact checking", "context_recall"),
    ("fact checking", "answer_relevancy"),
    ("fact checking", "semantic_similarity"),
    ("fact checking", "trustworthiness"),
    ("fact checking", "nv_response_groundedness"),
    ("multiple choice", "answer_relevancy"),
]

for combination in combinations_to_ignore:
    baseline.loc[baseline['dataset'] == combination[0], combination[1]] = float('nan')
    comparing.loc[comparing['dataset'] == combination[0], combination[1]] = float('nan')

## Calculate results

In [ ]:
baseline_overall = baseline[metrics].mean().mean()
comparing_overall = comparing[metrics].mean().mean()

baseline_by_dataset = baseline.groupby('dataset')[metrics].mean().T.mean()
comparing_by_dataset = comparing.groupby('dataset')[metrics].mean().T.mean()

baseline_detail = baseline.groupby('dataset')[metrics].mean()
baseline_by_metric = baseline[metrics].mean()

comparing_detail = comparing.groupby('dataset')[metrics].mean()
comparing_by_metric = comparing[metrics].mean()


## Overall Score

In [ ]:
comparison_string = f"baseline: {np.round(baseline_overall,3)}, "\
    f"comparing: {np.round(comparing_overall, 3)}, "\
    f"delta of "

fig = go.Figure(go.Indicator(
    mode = "number+delta",
    value = np.round(comparing_overall,3),
    number = {'prefix': f"Overall changed from {np.round(baseline_overall,3)} to "},
    delta = {'position': "top", 'reference': np.round(baseline_overall,3)}
))

fig.show()

## Detailed comparison

In [ ]:
def plot_polar_comparison(
    angular_values, baseline_radial, comparing_radial
):
    fig = go.Figure()

    fig.add_trace(
        go.Scatterpolar(
            r=baseline_radial,
            theta=angular_values,
            name="baseline",
            line=dict(color="grey")
        )
    )

    fig.add_trace(
        go.Scatterpolar(
            r=comparing_radial,
            theta=angular_values,
            name="comparing",
            line=dict(color='#0EA4E3')
        )
    )

    fig.update_layout(
        polar=dict(
            radialaxis=dict(
                visible=True, 
                range=[0, 1],
            )
        )
    )

    fig.show()

In [ ]:
metrics = baseline_by_metric.index.tolist()
metrics.append(baseline_by_metric.index.tolist()[0])

baseline_values = baseline_by_metric.values.tolist()
baseline_values.append(baseline_by_metric.values.tolist()[0])

comparing_values = comparing_by_metric.values.tolist()
comparing_values.append(comparing_by_metric.values.tolist()[0])

plot_polar_comparison(metrics, baseline_values, comparing_values)

In [ ]:
datasets = baseline_by_dataset.index.tolist()
datasets.append(baseline_by_dataset.index.tolist()[0])

baseline_values = baseline_by_dataset.values.tolist()
baseline_values.append(baseline_by_dataset.values.tolist()[0])

comparing_values = comparing_by_dataset.values.tolist()
comparing_values.append(comparing_by_dataset.values.tolist()[0])

plot_polar_comparison(datasets, baseline_values, comparing_values)

In [ ]:
import plotly.express as px

heatmap_data = comparing_detail - baseline_detail

display(baseline_detail)
display(comparing_detail)

text_matrix = heatmap_data.round(2).astype(str)
text_matrix = text_matrix.replace("nan", "n.a.")

fig = go.Figure(data=go.Heatmap(
    z=heatmap_data,
    x=heatmap_data.columns,
    y=heatmap_data.index,
    text=text_matrix,
    texttemplate="%{text}",
    colorscale=[(0.00, "#FF7B52"), (0.5, "#F1FF53"), (1.00, "#56D752")],
))

fig.update_layout(
    title="Mean Scores by Metric and Dataset",
    xaxis_title="Metrics",
    yaxis_title="Datasets",
    xaxis_nticks=len(heatmap_data.columns),
    yaxis_nticks=len(heatmap_data.index),
)

fig.show()